In [7]:
import pandas as pd
import numpy as np
from itertools import combinations
from statsmodels.stats.inter_rater import fleiss_kappa

# ==================== 配置区 ====================
# 模型名称与文件路径映射，可以放多个模型
MODEL_FILES = {
    # "gemini": "./gemini-3-flash-preview/new_annotated_data.csv",
    "deepseek": "./deepseek-v4-flash/new_annotated_data.csv",
    # "glm": "./glm-4/new_annotated_data.csv",
    "gpt": "./gpt-4o/new_annotated_data.csv",
    # "grok": "./grok-4-fast/new_annotated_data.csv",
    # "qwen": "./qwen-plus-2025-07-28/new_annotated_data.csv",
    # "mimo": "./mimo-v2-flash/new_annotated_data.csv"
}

DIMENSIONS = [
    "hate_speech", "false_info", "violence",
    "harassment", "obscenity", "illegal", "national_security"
]

VOTE_THRESHOLD = 0.5  # 多数投票比例，>0.5 即判 1（若偶数模型可用 0.5 平票自定义策略）
OUTPUT_FILE = "annotation_comparison.csv"
# ================================================

# 1. 读取并合并所有模型数据
dfs = {}
for model_name, path in MODEL_FILES.items():
    df = pd.read_csv(path)
    # 保留 index, text 和七个维度列，其余忽略（也可保留 explanations）
    cols_to_keep = ['index', 'text'] + DIMENSIONS
    dfs[model_name] = df[cols_to_keep].set_index('index')

# 对齐：按 index 取交集（或并集，根据需求）
common_indices = sorted(set.intersection(*[set(df.index) for df in dfs.values()]))
print(f"共有 {len(common_indices)} 条数据在所有模型中都存在。")

# 构建大宽表：每列如 hate_speech_A, hate_speech_B ...
combined = pd.DataFrame(index=common_indices)
combined['text'] = dfs[list(MODEL_FILES.keys())[0]].loc[common_indices, 'text']

for model_name, df in dfs.items():
    for dim in DIMENSIONS:
        combined[f"{dim}_{model_name}"] = df.loc[common_indices, dim].astype(int)

# 2. 投票集成
for dim in DIMENSIONS:
    # 取出该维度所有模型的列
    dim_cols = [col for col in combined.columns if col.startswith(f"{dim}_")]
    # 每行求平均
    avg = combined[dim_cols].mean(axis=1)
    # 多数投票（>0.5 为 1，等于 0.5 可设为 0 或随机）
    combined[f"{dim}_voted"] = (avg > VOTE_THRESHOLD).astype(int)

# 3. 一致性分析
print("\n========== 整体一致性（至少一个维度为1）==========")
# 每个模型先判定“该条数据是否违规”（任意维度为1）
for model_name in MODEL_FILES:
    dim_cols = [f"{dim}_{model_name}" for dim in DIMENSIONS]
    combined[f"any_risk_{model_name}"] = combined[dim_cols].max(axis=1)

risk_cols = [f"any_risk_{m}" for m in MODEL_FILES]
avg_risk = combined[risk_cols].mean(axis=1)
combined["any_risk_voted"] = (avg_risk > VOTE_THRESHOLD).astype(int)

# 完全一致的数据比例
all_equal = combined[risk_cols].eq(combined[risk_cols[0]], axis=0).all(axis=1)
print(f"所有模型完全一致的样本比例: {all_equal.mean():.3f}")

# 投票后，“有风险”的样本占比
print(f"投票后风险样本比例: {combined['any_risk_voted'].mean():.3f}")

# 4. 多评估者一致性（Fleiss' Kappa 逐维度）
print("\n========== Fleiss' Kappa（逐维度）==========")
for dim in DIMENSIONS:
    dim_cols = [f"{dim}_{m}" for m in MODEL_FILES]
    # 转换为评分者-类别计数矩阵（每个样本一行，列数=类别数（2类））
    # Fleiss' Kappa 需要格式: 每行是 [类别0的评分数, 类别1的评分数]
    fleiss_data = []
    for idx in combined.index:
        row = combined.loc[idx, dim_cols].values
        count1 = row.sum()          # 判为1的模型数
        count0 = len(row) - count1   # 判为0的模型数
        fleiss_data.append([count0, count1])
    fleiss_data = np.array(fleiss_data)
    try:
        kappa = fleiss_kappa(fleiss_data)
        print(f"{dim:20s}: Fleiss' Kappa = {kappa:.4f}")
    except Exception as e:
        print(f"{dim:20s}: 计算失败 ({e})")

# 5. 成对 Cohen's Kappa（任何两个模型）
print("\n========== 成对 Cohen's Kappa（整体风险）==========")
def cohen_kappa(a, b):
    """简易 Cohen's Kappa 计算（二分类）"""
    n = len(a)
    po = (a == b).mean()
    p1_a = a.mean()
    p1_b = b.mean()
    pe = p1_a * p1_b + (1 - p1_a) * (1 - p1_b)
    if pe == 1:
        return 1.0
    return (po - pe) / (1 - pe)

model_names = list(MODEL_FILES.keys())
for m1, m2 in combinations(model_names, 2):
    col1 = f"any_risk_{m1}"
    col2 = f"any_risk_{m2}"
    k = cohen_kappa(combined[col1].values, combined[col2].values)
    print(f"{m1} vs {m2}: Cohen's Kappa = {k:.4f}")
# 6. 保存汇总表
# 标记“争议样本”（模型不完全一致）
combined["disagreement"] = (~all_equal).astype(int)

# 修正：只包含实际存在的列
output_cols = ['text']
for model_name in MODEL_FILES:
    output_cols += [f"{dim}_{model_name}" for dim in DIMENSIONS]
output_cols += [f"{dim}_voted" for dim in DIMENSIONS]
output_cols += ['disagreement']

combined[output_cols].to_csv(OUTPUT_FILE, index=True, encoding='utf-8-sig')
print(f"\n比较结果已保存至 {OUTPUT_FILE}")
# 6. 保存汇总表（包含各模型原始预测 + 投票标签）
# 可选：标记“争议样本”（模型不完全一致）
# combined["disagreement"] = (~all_equal).astype(int)

# # 只保留需要的列输出（可自定义）
# output_cols = ['text'] + DIMENSIONS
# for model_name in MODEL_FILES:
#     output_cols += [f"{dim}_{model_name}" for dim in DIMENSIONS]
# output_cols += [f"{dim}_voted" for dim in DIMENSIONS]
# output_cols += ['disagreement']

# combined[output_cols].to_csv(OUTPUT_FILE, index=True, encoding='utf-8-sig')
# print(f"\n比较结果已保存至 {OUTPUT_FILE}")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


共有 27221 条数据在所有模型中都存在。

========== 整体一致性（至少一个维度为1）==========
所有模型完全一致的样本比例: 0.869
投票后风险样本比例: 0.368

========== Fleiss' Kappa（逐维度）==========
hate_speech         : Fleiss' Kappa = 0.6669
false_info          : Fleiss' Kappa = 0.2998
violence            : Fleiss' Kappa = 0.7020
harassment          : Fleiss' Kappa = 0.4967
obscenity           : Fleiss' Kappa = 0.7134
illegal             : Fleiss' Kappa = 0.5632
national_security   : Fleiss' Kappa = 0.5385

========== 成对 Cohen's Kappa（整体风险）==========
deepseek vs gpt: Cohen's Kappa = 0.7346

比较结果已保存至 annotation_comparison.csv


In [1]:
import pandas as pd
import numpy as np

# ==================== 配置区 ====================
# 模型文件路径映射（可增减模型数量）
MODEL_FILES = {
    "gemini": "./gemini-3-flash-preview/new_annotated_data.csv",
    "deepseek": "./deepseek-v4-flash/new_annotated_data.csv",
    "glm": "./glm-4/new_annotated_data.csv",
    "gpt": "./gpt-4o/new_annotated_data.csv",
    "grok": "./grok-4-fast/new_annotated_data.csv",
    # "qwen": "./qwen-plus-2025-07-28/new_annotated_data.csv",
    # "mimo": "./mimo-v2-flash/new_annotated_data.csv"
}

DIMENSIONS = [
    "hate_speech", "false_info", "violence",
    "harassment", "obscenity", "illegal", "national_security"
]

VOTE_THRESHOLD = 0.5        # 半数以上判1，等于0.5时按你的需求可调（如 >=0.5）
OUTPUT_FILE = "voted_final_labels.csv"
# ================================================

# 1. 读取所有模型数据，只保留 index, text 和七个维度
dfs = {}
for model_name, path in MODEL_FILES.items():
    df = pd.read_csv(path)
    cols_needed = ['index', 'text'] + DIMENSIONS
    dfs[model_name] = df[cols_needed].set_index('index')

# 2. 按 index 取交集（所有模型都有的样本）
common_indices = sorted(set.intersection(*[set(df.index) for df in dfs.values()]))
print(f"共有 {len(common_indices)} 条数据在所有模型中都有标注。")

# 3. 构建投票所需的数据：对于每个样本、每个维度，收集所有模型的分数
# 采用宽表形式，列如 hate_speech_gemini, hate_speech_deepseek 等
combined = pd.DataFrame(index=common_indices)
combined['text'] = dfs[list(MODEL_FILES.keys())[0]].loc[common_indices, 'text']

for model_name, df in dfs.items():
    for dim in DIMENSIONS:
        combined[f"{dim}_{model_name}"] = df.loc[common_indices, dim].astype(int)

# 4. 多数投票
for dim in DIMENSIONS:
    dim_cols = [f"{dim}_{model}" for model in MODEL_FILES]
    # 每行判1的模型比例
    avg = combined[dim_cols].mean(axis=1)
    combined[f"{dim}_voted"] = (avg > VOTE_THRESHOLD).astype(int)

# 5. 构造最终输出结果
result = pd.DataFrame()
result['index'] = common_indices
result['text'] = combined['text'].values

for dim in DIMENSIONS:
    result[dim] = combined[f"{dim}_voted"].values

# 整体风险（任一维度为1）
result['overall_risk'] = result[DIMENSIONS].max(axis=1)

# 可选：标记是否有模型分歧（至少一个维度所有模型不完全一致）
# 计算每个维度是否所有模型一致
disagreement_flags = np.zeros(len(common_indices), dtype=bool)
for dim in DIMENSIONS:
    dim_cols = [f"{dim}_{m}" for m in MODEL_FILES]
    # 该维度所有模型取值不全相等则为分歧
    not_equal = combined[dim_cols].nunique(axis=1) > 1
    disagreement_flags = disagreement_flags | not_equal.values
result['disagreement'] = disagreement_flags.astype(int)

# 6. 保存
result.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"最终投票标签已保存至 {OUTPUT_FILE}")
print(f"其中风险样本占比: {result['overall_risk'].mean():.2%}")
print(f"存在模型分歧的样本占比: {result['disagreement'].mean():.2%}")

共有 2686 条数据在所有模型中都有标注。
最终投票标签已保存至 voted_final_labels.csv
其中风险样本占比: 49.52%
存在模型分歧的样本占比: 66.42%


In [3]:
import pandas as pd

# ========== 配置 ==========
CSV_PATH = "./deepseek-v4-flash/new_annotated_data.csv"   # 模型CSV路径
DIMENSIONS = [
    "hate_speech", "false_info", "violence",
    "harassment", "obscenity", "illegal", "national_security"
]
# =========================

df = pd.read_csv(CSV_PATH)

print("===== 各维度命中（为1）的样本数 =====")
for dim in DIMENSIONS:
    count = int(df[dim].sum())
    print(f"{dim:20s}: {count:5d} 条")

# 任意维度命中（即整体违规）的样本数
df['any_risk'] = df[DIMENSIONS].max(axis=1)
total_risky = int(df['any_risk'].sum())
total_safe = len(df) - total_risky
print(f"\n整体违规样本数: {total_risky}")
print(f"安全样本数: {total_safe}")

# 多标签组合 Top 10（看哪些组合最常见）
from itertools import combinations
import json

# 将7维分数转为二元元组作为组合键
def get_combination(row):
    return "|".join([dim for dim in DIMENSIONS if row[dim] == 1])

df['combo'] = df.apply(get_combination, axis=1)
print("\n===== 命中组合 Top 10 =====")
print(df['combo'].value_counts().head(10).to_string())

===== 各维度命中（为1）的样本数 =====
hate_speech         :  5275 条
false_info          :   405 条
violence            :  1315 条
harassment          :  4317 条
obscenity           :  2369 条
illegal             :   202 条
national_security   :  1441 条

整体违规样本数: 10960
安全样本数: 16262

===== 命中组合 Top 10 =====
combo
                                          16262
hate_speech                                2821
harassment                                 2292
obscenity                                  1412
hate_speech|harassment                      873
national_security                           638
harassment|obscenity                        544
hate_speech|violence                        483
hate_speech|national_security               224
hate_speech|violence|national_security      201


In [4]:
import pandas as pd

# ========== 配置 ==========
CSV_PATH = "./gpt-4o/new_annotated_data.csv"   # 模型CSV路径
DIMENSIONS = [
    "hate_speech", "false_info", "violence",
    "harassment", "obscenity", "illegal", "national_security"
]
# =========================

df = pd.read_csv(CSV_PATH)

print("===== 各维度命中（为1）的样本数 =====")
for dim in DIMENSIONS:
    count = int(df[dim].sum())
    print(f"{dim:20s}: {count:5d} 条")

# 任意维度命中（即整体违规）的样本数
df['any_risk'] = df[DIMENSIONS].max(axis=1)
total_risky = int(df['any_risk'].sum())
total_safe = len(df) - total_risky
print(f"\n整体违规样本数: {total_risky}")
print(f"安全样本数: {total_safe}")

# 多标签组合 Top 10（看哪些组合最常见）
from itertools import combinations
import json

# 将7维分数转为二元元组作为组合键
def get_combination(row):
    return "|".join([dim for dim in DIMENSIONS if row[dim] == 1])

df['combo'] = df.apply(get_combination, axis=1)
print("\n===== 命中组合 Top 10 =====")
print(df['combo'].value_counts().head(10).to_string())

===== 各维度命中（为1）的样本数 =====
hate_speech         :  7667 条
false_info          :   326 条
violence            :  1282 条
harassment          :  2747 条
obscenity           :  3399 条
illegal             :   190 条
national_security   :   829 条

整体违规样本数: 12634
安全样本数: 14587

===== 命中组合 Top 10 =====
combo
                                    14587
hate_speech                          5178
obscenity                            2214
harassment                            966
hate_speech|harassment                868
hate_speech|violence                  537
national_security                     482
harassment|obscenity                  370
violence                              340
hate_speech|harassment|obscenity      336


In [6]:
import pandas as pd
import numpy as np
from itertools import combinations

# ==================== 配置区 ====================
MODEL_FILES = {
    # "gemini": "./gemini-3-flash-preview/new_annotated_data.csv",
    "deepseek": "./deepseek-v4-flash/new_annotated_data.csv",
    # "glm": "./glm-4/new_annotated_data.csv",
    "gpt": "./gpt-4o/new_annotated_data.csv",
    # "grok": "./grok-4-fast/new_annotated_data.csv",
    # "qwen": "./qwen-plus-2025-07-28/new_annotated_data.csv",
    # "mimo": "./mimo-v2-flash/new_annotated_data.csv"
}

DIMENSIONS = [
    "hate_speech", "false_info", "violence",
    "harassment", "obscenity", "illegal", "national_security"
]
# ================================================

# 1. 读取并准备每个模型的二分类序列
risk_scores = {}
for model_name, path in MODEL_FILES.items():
    df = pd.read_csv(path)
    # 计算一条数据是否不安全（任一维度为1）
    risk = df[DIMENSIONS].max(axis=1).astype(int)
    risk_scores[model_name] = risk
    # 计算风险比例
    risk_rate = risk.mean()
    print(f"{model_name:10s}: 不安全样本占比 = {risk_rate:.2%} ({risk.sum()}/{len(risk)})")

# 2. 对齐数据（取所有模型共有的 index）
# 假设所有模型的数据行数一致且顺序一致（通常都是从同一份原始数据标注的）
# 若不保证对齐，则以某一模型为基准按 index 对齐，这里简化处理假设都用相同行数且顺序相同
# 稳健做法：按 index 取交集
# 先读取出所有 index
all_indices = []
for model_name, path in MODEL_FILES.items():
    df_idx = pd.read_csv(path, usecols=['index'])
    all_indices.append(set(df_idx['index']))
common_indices = sorted(set.intersection(*all_indices))

# 重新读取并对齐
aligned = {}
for model_name, path in MODEL_FILES.items():
    df = pd.read_csv(path)
    df = df.set_index('index').loc[common_indices]
    risk = df[DIMENSIONS].max(axis=1).astype(int)
    aligned[model_name] = risk.values

n = len(common_indices)
print(f"\n对齐后共 {n} 条数据")

# 3. 计算成对 Cohen's Kappa
def cohen_kappa(y1, y2):
    n = len(y1)
    po = np.mean(y1 == y2)
    p1_a = np.mean(y1)
    p1_b = np.mean(y2)
    pe = p1_a * p1_b + (1 - p1_a) * (1 - p1_b)
    if pe == 1:
        return 1.0
    return (po - pe) / (1 - pe)

print("\n========== 成对一致性 (Cohen's Kappa) ==========")
model_names = list(MODEL_FILES.keys())
for m1, m2 in combinations(model_names, 2):
    k = cohen_kappa(aligned[m1], aligned[m2])
    print(f"{m1:10s} vs {m2:10s}: Kappa = {k:.4f}")

# 4. 完全一致率（所有模型标注完全相同的样本比例）
all_arrays = np.column_stack([aligned[m] for m in model_names])
fully_agree = np.mean(np.all(all_arrays == all_arrays[:, 0:1], axis=1))
print(f"\n所有模型完全一致的样本比例: {fully_agree:.2%}")

deepseek  : 不安全样本占比 = 40.26% (10960/27222)
gpt       : 不安全样本占比 = 46.41% (12634/27221)

对齐后共 27221 条数据

========== 成对一致性 (Cohen's Kappa) ==========
deepseek   vs gpt       : Kappa = 0.7346

所有模型完全一致的样本比例: 86.91%
